# Argus Vision — Agent A Training (EfficientNet-B4)

Adversarial multi-agent visual debate for uncertainty-aware medical image classification.

This notebook trains **Agent A**, an EfficientNet-B4 convolutional classifier, on the
ISIC-8 dermoscopic lesion taxonomy:

`["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]`

Pipeline:
1. Environment + ISIC 2019 dataset download (Kaggle).
2. Exploratory data analysis (class distribution + sample grid).
3. `ISICDataset` + class-weighted `WeightedRandomSampler` DataLoaders.
4. `timm` EfficientNet-B4 with a frozen backbone and a trainable 8-class head.
5. **Phase 1** — head-only training with `FocalLoss` + `AdamW`.
6. **Phase 2** — unfreeze the last two blocks and fine-tune at a lower LR.
7. Training curves (loss / balanced accuracy / macro AUC).
8. Save the best checkpoint to `../backend/checkpoints/agent_a_best.pth`.
9. Confusion-matrix heatmap on the held-out validation split.
10. Grad-CAM++ saliency over a 3×3 grid of validation images.

All hyper-parameters, transforms and the dataset class are imported from the shared
`config`, `dataset` and `losses` modules that live alongside this notebook.

## 1. Environment setup and ISIC 2019 download

Install the pinned training dependencies, register Kaggle credentials (replace the
placeholders with your own token), download the ISIC 2019 dataset and unzip it into the
local data directory defined in `config.py`.

In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

# Install the exact pinned training stack defined in requirements_training.txt.
subprocess.run(
    ["pip", "install", "-q", "-r", "requirements_training.txt"],
    check=True,
)

# Kaggle API credentials. Replace these placeholder values (or export the matching
# environment variables before launching Jupyter) with your own Kaggle token.
os.environ["KAGGLE_USERNAME"] = os.environ.get("KAGGLE_USERNAME", "your_kaggle_username")
os.environ["KAGGLE_KEY"] = os.environ.get("KAGGLE_KEY", "your_kaggle_key")

from config import AgentAConfig  # noqa: E402  (import after pip install)

CFG = AgentAConfig()
DATA_DIR = Path(CFG.data_dir)
DATA_DIR.mkdir(parents=True, exist_ok=True)

ZIP_PATH = DATA_DIR / f"{CFG.kaggle_dataset_slug.split('/')[-1]}.zip"

if not ZIP_PATH.exists():
    # Download the ISIC 2019 dermoscopy dataset from Kaggle into DATA_DIR.
    subprocess.run(
        [
            "kaggle",
            "datasets",
            "download",
            "-d",
            CFG.kaggle_dataset_slug,
            "-p",
            str(DATA_DIR),
        ],
        check=True,
    )
else:
    print(f"Archive already present at {ZIP_PATH}, skipping download.")

# Unzip every archive found in DATA_DIR (the Kaggle dataset may ship multiple zips).
for archive in sorted(DATA_DIR.glob("*.zip")):
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(DATA_DIR)
    print(f"Extracted {archive.name}")

print("Data directory contents:")
for entry in sorted(DATA_DIR.iterdir()):
    print(" -", entry.name)

## 2. Exploratory data analysis

Load the training ground-truth CSV with `pandas`, collapse the ISIC 2019 one-hot
columns into the canonical ISIC-8 label order, plot the class distribution and render a
3×3 grid of representative sample images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

from config import CLASS_NAMES, FULL_NAMES
from dataset import load_isic_dataframe

sns.set_theme(style="whitegrid")

# load_isic_dataframe reads the ISIC 2019 ground-truth CSV, maps the one-hot columns to
# the canonical ISIC-8 integer labels and returns absolute image paths.
df = load_isic_dataframe(CFG)
print(f"Loaded {len(df):,} labelled images.")
df.head()

In [ ]:
# Class-distribution bar chart over the eight ISIC classes.
counts = df["label"].value_counts().sort_index()
class_counts = pd.Series(
    [int(counts.get(idx, 0)) for idx in range(len(CLASS_NAMES))],
    index=CLASS_NAMES,
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=class_counts.index, y=class_counts.values, ax=ax, hue=class_counts.index, palette="viridis", legend=False)
ax.set_title("ISIC-8 class distribution (training split)")
ax.set_xlabel("Class")
ax.set_ylabel("Number of images")
for patch, value in zip(ax.patches, class_counts.values):
    ax.annotate(
        f"{int(value):,}",
        (patch.get_x() + patch.get_width() / 2.0, patch.get_height()),
        ha="center",
        va="bottom",
        fontsize=9,
    )
plt.tight_layout()
plt.show()

print("Per-class counts:")
for name, value in class_counts.items():
    print(f"  {name:5s} ({FULL_NAMES[name]:25s}): {int(value):,}")

In [ ]:
# 3x3 grid of one representative sample image per class (first eight classes plus one).
rng = np.random.default_rng(CFG.seed)
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for ax_idx, ax in enumerate(axes.ravel()):
    class_idx = ax_idx % len(CLASS_NAMES)
    subset = df[df["label"] == class_idx]
    if len(subset) == 0:
        ax.axis("off")
        continue
    sample_row = subset.iloc[int(rng.integers(0, len(subset)))]
    image = Image.open(sample_row["path"]).convert("RGB")
    ax.imshow(image)
    ax.set_title(f"{CLASS_NAMES[class_idx]} — {FULL_NAMES[CLASS_NAMES[class_idx]]}", fontsize=10)
    ax.axis("off")
plt.suptitle("ISIC-8 sample dermoscopic images", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Datasets, samplers and DataLoaders

Build a stratified train/validation split, instantiate the shared `ISICDataset` with the
EfficientNet-B4 training/validation transforms, compute inverse-frequency class weights
and wire a `WeightedRandomSampler` so the heavily imbalanced minority classes (DF, VASC,
SCC) are over-sampled during training.

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler

from dataset import ISICDataset, build_train_transforms, build_val_transforms

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)
print(f"Using device: {DEVICE}")

# Stratified split preserves the per-class ratios across train and validation.
train_df, val_df = train_test_split(
    df,
    test_size=CFG.val_fraction,
    stratify=df["label"],
    random_state=CFG.seed,
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

train_dataset = ISICDataset(train_df, transform=build_train_transforms(CFG.image_size))
val_dataset = ISICDataset(val_df, transform=build_val_transforms(CFG.image_size))

In [ ]:
# Inverse-frequency class weights, used both for FocalLoss alpha and the sampler.
train_labels = train_df["label"].to_numpy()
class_sample_counts = np.bincount(train_labels, minlength=len(CLASS_NAMES)).astype(np.float64)
class_sample_counts = np.clip(class_sample_counts, 1.0, None)

class_weights = class_sample_counts.sum() / (len(CLASS_NAMES) * class_sample_counts)
class_weights = class_weights / class_weights.sum() * len(CLASS_NAMES)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)
print("Class weights:", {name: round(float(w), 3) for name, w in zip(CLASS_NAMES, class_weights)})

# Per-sample weights drive the WeightedRandomSampler to balance the minibatches.
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(
    weights=torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    sampler=sampler,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE == "cuda"),
    drop_last=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    num_workers=CFG.num_workers,
    pin_memory=(DEVICE == "cuda"),
)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

## 4. Model: EfficientNet-B4 with a frozen backbone

Create an ImageNet-pretrained EfficientNet-B4 with an 8-class head via `timm`. Every
backbone parameter is frozen so Phase 1 trains only the classifier head.

In [ ]:
import timm
import torch.nn as nn

# The timm identifier MUST match backend/ml/agents/agent_a.py ("efficientnet_b4").
model = timm.create_model("efficientnet_b4", pretrained=True, num_classes=len(CLASS_NAMES))
model = model.to(DEVICE)

# timm exposes the classifier head via get_classifier(); freeze everything else so that
# Phase 1 optimizes only the 8-class head.
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params (head only): {trainable:,} / {total:,}")

In [ ]:
from typing import Dict, List, Tuple

import torch.nn.functional as F
from sklearn.metrics import balanced_accuracy_score
from torchmetrics.classification import MulticlassAUROC

# torchmetrics computes the macro one-vs-rest AUROC on-device for the eight classes.
auroc_metric = MulticlassAUROC(num_classes=len(CLASS_NAMES), average="macro").to(DEVICE)


@torch.no_grad()
def evaluate(network: nn.Module, loader: DataLoader) -> Tuple[float, float, np.ndarray, np.ndarray]:
    """Evaluate a network on a loader.

    Args:
        network: The classification model in any mode (set to eval internally).
        loader: A DataLoader yielding ``(image, label)`` batches.

    Returns:
        A tuple ``(balanced_accuracy, macro_auc, y_true, y_pred)`` where the
        metrics are floats and the arrays hold integer class indices.
    """
    network.eval()
    auroc_metric.reset()
    all_preds: List[np.ndarray] = []
    all_targets: List[np.ndarray] = []
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        probs = F.softmax(network(images), dim=1)
        auroc_metric.update(probs, targets)
        all_preds.append(probs.argmax(dim=1).detach().cpu().numpy())
        all_targets.append(targets.detach().cpu().numpy())
    preds_arr = np.concatenate(all_preds, axis=0)
    targets_arr = np.concatenate(all_targets, axis=0)
    bal_acc = float(balanced_accuracy_score(targets_arr, preds_arr))
    macro_auc = float(auroc_metric.compute().item())
    return bal_acc, macro_auc, targets_arr, preds_arr


def run_epoch(
    network: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
) -> float:
    """Run a single optimization epoch and return the mean training loss.

    Args:
        network: The model to optimize in-place.
        loader: The training DataLoader.
        criterion: The loss function (FocalLoss).
        optimizer: The optimizer driving the parameter updates.

    Returns:
        The mean per-batch training loss for the epoch.
    """
    network.train()
    running_loss = 0.0
    num_batches = 0
    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        logits = network(images)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())
        num_batches += 1
    return running_loss / max(num_batches, 1)


# Containers shared across both training phases for the curve plots.
history: Dict[str, List[float]] = {"loss": [], "bal_acc": [], "macro_auc": []}
CHECKPOINT_PATH = Path(CFG.checkpoint_path)
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
best_macro_auc = -1.0

## 5. Phase 1 — head-only training

Train only the classifier head with `FocalLoss(alpha=class_weights)` and `AdamW(lr=LR_HEAD)`.
Each epoch logs validation balanced accuracy and macro one-vs-rest AUC and snapshots the
best checkpoint by macro AUC.

In [ ]:
from losses import FocalLoss

criterion = FocalLoss(alpha=class_weights_tensor, gamma=CFG.focal_gamma)

head_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.lr_head,
    weight_decay=CFG.weight_decay,
)

for epoch in range(1, CFG.max_epochs_head + 1):
    train_loss = run_epoch(model, train_loader, criterion, head_optimizer)
    bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
    history["loss"].append(train_loss)
    history["bal_acc"].append(bal_acc)
    history["macro_auc"].append(macro_auc)
    print(
        f"[P1 {epoch:02d}/{CFG.max_epochs_head}] "
        f"loss={train_loss:.4f} bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f}"
    )
    if not np.isnan(macro_auc) and macro_auc > best_macro_auc:
        best_macro_auc = macro_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"    ✓ new best macro AUC={best_macro_auc:.4f} -> saved {CHECKPOINT_PATH}")

## 6. Phase 2 — fine-tune the last two blocks

Unfreeze the final two EfficientNet stages (`blocks[-2:]`) together with the head and
fine-tune at the lower `LR_BACKBONE`. The best checkpoint by macro AUC continues to be
tracked across both phases.

In [ ]:
# timm EfficientNet exposes the inverted-residual stages as model.blocks (a Sequential
# of stage sub-sequences). Unfreeze the last two stages plus the classifier head.
for param in model.parameters():
    param.requires_grad = False
for param in model.get_classifier().parameters():
    param.requires_grad = True
for stage in list(model.blocks)[-2:]:
    for param in stage.parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (last 2 blocks + head): {trainable:,}")

finetune_optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=CFG.lr_backbone,
    weight_decay=CFG.weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    finetune_optimizer,
    T_max=CFG.max_epochs_finetune,
)

for epoch in range(1, CFG.max_epochs_finetune + 1):
    train_loss = run_epoch(model, train_loader, criterion, finetune_optimizer)
    scheduler.step()
    bal_acc, macro_auc, _, _ = evaluate(model, val_loader)
    history["loss"].append(train_loss)
    history["bal_acc"].append(bal_acc)
    history["macro_auc"].append(macro_auc)
    print(
        f"[P2 {epoch:02d}/{CFG.max_epochs_finetune}] "
        f"loss={train_loss:.4f} bal_acc={bal_acc:.4f} macro_auc={macro_auc:.4f}"
    )
    if not np.isnan(macro_auc) and macro_auc > best_macro_auc:
        best_macro_auc = macro_auc
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"    ✓ new best macro AUC={best_macro_auc:.4f} -> saved {CHECKPOINT_PATH}")

print(f"Best macro AUC across both phases: {best_macro_auc:.4f}")

## 7. Training curves

Plot the training loss, validation balanced accuracy and validation macro AUC over the
combined Phase 1 + Phase 2 schedule. The dashed line marks the Phase 1 → Phase 2 boundary.

In [ ]:
epochs_axis = np.arange(1, len(history["loss"]) + 1)
phase_boundary = CFG.max_epochs_head + 0.5

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
panels = [
    ("loss", "Training loss", "tab:red"),
    ("bal_acc", "Val balanced accuracy", "tab:blue"),
    ("macro_auc", "Val macro AUC", "tab:green"),
]
for ax, (key, title, color) in zip(axes, panels):
    ax.plot(epochs_axis, history[key], marker="o", color=color)
    ax.axvline(phase_boundary, linestyle="--", color="gray", alpha=0.7, label="P1→P2")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.legend()
plt.suptitle("Agent A (EfficientNet-B4) training curves", fontsize=14)
plt.tight_layout()
plt.show()

## 8. Save the best checkpoint

Reload the best weights observed during training (already serialized whenever macro AUC
improved) and re-save them to the canonical backend path
`../backend/checkpoints/agent_a_best.pth` consumed by `AgentA` at inference time.

In [ ]:
BACKEND_CHECKPOINT = Path("../backend/checkpoints/agent_a_best.pth")
BACKEND_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

# Restore the best state dict captured during training, then persist it where the
# backend AgentA loader expects it.
best_state = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(best_state)
torch.save(model.state_dict(), BACKEND_CHECKPOINT)
print(f"Saved best Agent A checkpoint to {BACKEND_CHECKPOINT.resolve()}")

## 9. Confusion matrix

Run the best model over the validation split and render a row-normalized per-class
confusion-matrix heatmap with `seaborn`.

In [ ]:
from sklearn.metrics import confusion_matrix

_, _, y_true, y_pred = evaluate(model, val_loader)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(CLASS_NAMES))))
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, np.clip(row_sums, 1, None), dtype=np.float64)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_norm,
    annot=True,
    fmt=".2f",
    cmap="magma",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax,
    cbar_kws={"label": "Row-normalized frequency"},
)
ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")
ax.set_title("Agent A — validation confusion matrix (row-normalized)")
plt.tight_layout()
plt.show()

## 10. Grad-CAM++ explainability

Visualize where Agent A attends using **Grad-CAM++** from `pytorch_grad_cam`. The CAM
target layer is the final convolutional block of the EfficientNet backbone
(`model.conv_head`). We overlay the saliency on a 3×3 grid of validation images.

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from config import IMAGENET_MEAN, IMAGENET_STD

model.eval()
# conv_head is the last conv layer before pooling in timm EfficientNet — the canonical
# Grad-CAM target for this architecture.
target_layers = [model.conv_head]
mean = np.array(IMAGENET_MEAN, dtype=np.float32)
std = np.array(IMAGENET_STD, dtype=np.float32)

grid_rows = val_df.sample(n=9, random_state=CFG.seed).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
with GradCAMPlusPlus(model=model, target_layers=target_layers) as cam:
    for ax, (_, row) in zip(axes.ravel(), grid_rows.iterrows()):
        # Build the normalized input tensor and the [0, 1] RGB float image for overlay.
        pil_image = Image.open(row["path"]).convert("RGB").resize((CFG.image_size, CFG.image_size))
        rgb = np.asarray(pil_image, dtype=np.float32) / 255.0
        norm = (rgb - mean) / std
        input_tensor = torch.from_numpy(norm.transpose(2, 0, 1)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_idx = int(torch.softmax(model(input_tensor), dim=1).argmax(dim=1).item())
        grayscale_cam = cam(
            input_tensor=input_tensor,
            targets=[ClassifierOutputTarget(pred_idx)],
        )[0]
        overlay = show_cam_on_image(rgb, grayscale_cam, use_rgb=True)
        ax.imshow(overlay)
        true_idx = int(row["label"])
        ax.set_title(
            f"true={CLASS_NAMES[true_idx]} / pred={CLASS_NAMES[pred_idx]}",
            fontsize=10,
            color=("green" if pred_idx == true_idx else "red"),
        )
        ax.axis("off")
plt.suptitle("Agent A — Grad-CAM++ saliency", fontsize=14)
plt.tight_layout()
plt.show()